**Match Histone Modifier Loss Samples to REdiscoverTE Samples**

This notebook takes the pancancer histone modifier loss cohort from Notebook 01 and checks which samples are present in the REdiscoverTE ExpressionSet.

The output is the case table you will use for WT matching.

In [ ]:
# Load libraries and set paths

library(dplyr)
library(readr)
library(tibble)
library(Biobase)
library(stringr)
library(purrr)

setwd("/home/kennes38/ResearchProject/13_histone_modifier_TEdata_matching")

gene_set_label <- "histone_modifier_loss"


loss_input_dir <- "../02_histone_modifier_LoFHomDel_finalcohort"

matching_output_dir <- "."

tedata_root <- "../TEdata_resources"

case_file <- file.path(
  loss_input_dir,
  paste0("pancancer_", gene_set_label, "_patients.csv")
)

file.exists(case_file)
dir.exists(tedata_root)
dir.exists(matching_output_dir)

In [ ]:
# Load REdiscoverTE ExpressionSet

rds_hits <- list.files(
  tedata_root,
  pattern = "eset_TCGA_TE_intergenic_logCPM\\.RDS$",
  recursive = TRUE,
  full.names = TRUE
)

rds_hits
stopifnot(length(rds_hits) == 1)

dat <- readRDS(rds_hits[1])

class(dat)
length(sampleNames(dat))
head(sampleNames(dat))

In [ ]:
# Build REdiscoverTE sample table

tedata_samples <- tibble(
  TEdata_full_id = sampleNames(dat),

  # sample_id_16 is the best sample-level TCGA match key.
  sample_id_16 = substr(sampleNames(dat), 1, 16),

  # patient_id is useful for checking duplicate patient use.
  patient_id = substr(sampleNames(dat), 1, 12),

  # 01 = primary tumour
  sample_type_code = substr(sample_id_16, 14, 15)
)

tedata_samples %>%
  count(sample_type_code, sort = TRUE)

In [ ]:
# Load Histone Modifier loss cases

loss_cases <- read_csv(case_file, show_col_types = FALSE) %>%
  mutate(
    sample_id_16 = substr(as.character(sample_id_16), 1, 16),
    patient_id = substr(as.character(patient_id), 1, 12),
    case_sample_type_code = substr(sample_id_16, 14, 15)
  ) %>%
  distinct()

cat("Loss samples:", nrow(loss_cases), "\n")

loss_cases %>%
  count(project, sort = TRUE)

loss_cases %>%
  count(case_sample_type_code, sort = TRUE)

In [ ]:
# Match loss cases to REdiscoverTe by sample barcode

loss_with_tedata_match <- loss_cases %>%
  # Match by exact 16-character sample barcode
  left_join(
    tedata_samples %>%
      select(
        TEdata_full_id,
        TEdata_sample_id_16 = sample_id_16,
        TEdata_patient_id = patient_id,
        TEdata_sample_type_code = sample_type_code
      ),
    by = c("sample_id_16" = "TEdata_sample_id_16")
  ) %>%
  mutate(
    has_TEdata_match = !is.na(TEdata_full_id),
    has_rna_sample_match = has_TEdata_match
  )

loss_with_tedata_match %>%
  count(has_TEdata_match)

loss_with_tedata_match %>%
  filter(has_TEdata_match) %>%
  count(project, sort = TRUE)

In [ ]:
# Save TE-ready case table

te_ready_cases <- loss_with_tedata_match %>%
  # Only samples present in REdiscoverTE can go forward to WT matching and DE.
  filter(has_TEdata_match) %>%
  arrange(project, sample_id_16)

write_csv(
  loss_with_tedata_match,
  file.path(matching_output_dir, paste0("pancancer_", gene_set_label, "_samples_with_TEdata_match.csv"))
)

write_csv(
  te_ready_cases,
  file.path(matching_output_dir, paste0("pancancer_", gene_set_label, "_samples_with_RNA_match_TE_ready.csv"))
)

cat("TE-ready loss samples:", nrow(te_ready_cases), "\n")
cat("TE-ready unique patients:", n_distinct(te_ready_cases$patient_id), "\n")